In [ ]:
from sleep_stage_prediction.data import load_dreamt, Workflow
from sleep_stage_prediction.models import ConvolutionalClassifier
import numpy as np 
import torch 
import torch.nn as nn 

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
frequency = 64 
seed = 42
nb_patients = 10 
workflow = Workflow.CENTRALIZED
lr = 0.001 
epochs = 1
batch_size = 128
momentum = 0.9 

X_train, X_test, y_train, y_test = load_dreamt(
        nb_patients, workflow=workflow, frequency=frequency, seed=seed
    )


In [ ]:
in_samples = np.random.choice(X_train.shape[0], size=X_train.shape[0] // 2 )
X_in = X_train[in_samples]
y_in = y_train[in_samples]    

In [ ]:
from sleep_stage_prediction.models import train_model, test_model


model = ConvolutionalClassifier(channel_in=7, kernel_size=7).to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
criterion = nn.CrossEntropyLoss(reduction="sum")

train_model(model, X_in, y_in, optimizer, criterion, epochs, batch_size, DEVICE, seed)
test_model(model, X_test, y_test, criterion, device=DEVICE)


In [ ]:
def MPE(confidence_vector, label):
    _mpe = 0
    for j in range(confidence_vector.shape[1]):
        if j != label:
            _mpe += confidence_vector[j] * torch.log(1 - confidence_vector[j])

    _mpe = -1 * _mpe + (1  - confidence_vector[label]) * torch.log(confidence_vector[label])

    return _mpe 

In [ ]:
input = torch.Tensor(X_in[0]).permute(1,0)
print(input.shape)
model.to("cpu")

confidence_vector = model(input)